# 10 -- GPT-5.1 Explanation Quality Audit

**Purpose:** Go beyond structural/schema validation to verify GPT-5.1 explanations
are genuinely grounded in log evidence, not hallucinated content that merely passes
verification rules.

**Approach (inspired by 09_human_evaluation):**
1. Load GPT-5.1 results + raw HDFS log lines
2. **Span Verification** -- do cited evidence spans (e.g. `E0-L17`) actually exist and match claim content?
3. **Claim-Log Alignment** -- do observation claims describe what the logs actually say?
4. **Hallucination Detection** -- flag claims that reference non-existent content or fabricate log messages
5. **Human Review Interface** -- display session with log lines for manual confirmation

**Key Question:** Are the perfect scores (C=4.50, Co=5.00, E=5.00, Verif=100%) genuine or structurally gamed?

**Rubric (same as 09):**

| Dimension | 1 | 3 | 5 |
|-----------|---|---|---|
| **Correctness** | Contradicts the logs | Partially correct | Fully accurate |
| **Completeness** | Major signals missing | Main signal covered | All signals addressed |
| **Evidence Grounding** | Wrong or no evidence | Most claims grounded | Every claim backed by correct span |

| Binary | Y | N |
|--------|---|---|
| **Actionable** | Engineer could act on this | Too vague or wrong |

In [1]:
# === Section 1: Imports and Configuration ===
import sys, json, re, math, random
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# === File paths ===
GPT51_RESULTS = PROJECT_ROOT / 'results_HDFS' / 'model_cmp_gpt-5p1_20260301_064148.jsonl'
LLAMA_RESULTS = PROJECT_ROOT / 'results_HDFS' / 'model_cmp_llama3p1_8b_20260301_062818.jsonl'
AUDIT_RATINGS_PATH = PROJECT_ROOT / 'results_HDFS' / 'gpt51_audit_ratings.json'

assert GPT51_RESULTS.exists(), f"GPT-5.1 results not found: {GPT51_RESULTS}"
assert LLAMA_RESULTS.exists(), f"llama3.1 results not found: {LLAMA_RESULTS}"

print(f"Project root: {PROJECT_ROOT}")
print(f"[OK] Imports loaded")

Project root: /home/dave/agentic-log-explanations
[OK] Imports loaded


## 2. Load Explanations & Raw Log Lines

Load GPT-5.1 and llama3.1:8b results (multi-line JSONL), then load raw HDFS log lines
for each session so we can verify claims against ground truth.

In [2]:
# === Load multi-line JSONL results ===

def load_multiline_jsonl(path):
    """Parse multi-line pretty-printed JSONL."""
    with open(path) as f:
        content = f.read()
    objects = []
    buf = []; depth = 0
    for line in content.split('\n'):
        stripped = line.strip()
        if not stripped:
            continue
        depth += stripped.count('{') - stripped.count('}')
        depth += stripped.count('[') - stripped.count(']')
        buf.append(line)
        if depth <= 0 and buf:
            try:
                objects.append(json.loads('\n'.join(buf)))
            except json.JSONDecodeError:
                pass
            buf = []
            depth = 0
    return objects

gpt51_records = load_multiline_jsonl(GPT51_RESULTS)
llama_records = load_multiline_jsonl(LLAMA_RESULTS)

print(f"[OK] GPT-5.1: {len(gpt51_records)} records")
print(f"[OK] llama3.1:8b: {len(llama_records)} records")

# Build lookup by session_id
gpt51_by_sid = {r['session_id']: r for r in gpt51_records}
llama_by_sid = {r['session_id']: r for r in llama_records}

[OK] GPT-5.1: 24 records
[OK] llama3.1:8b: 24 records


In [34]:
# === Load raw HDFS log lines for the 24 edge-case sessions + all evidence sessions ===
# This takes ~3-5 minutes on first run (1.5 GB HDFS.log)

from src.data_loader import HDFSDataLoader

# Target sessions: the 24 GPT-5.1 evaluation sessions
target_sids = set(gpt51_by_sid.keys())

# Evidence sessions: E1-E5 referenced in each record's evidence_id_mapping.
# They are stored as "E_HDFS_blk_xxx" in the mapping; strip the E_ prefix
# to get the actual session ID as it appears in the HDFS log.
evidence_sids = set()
for r in gpt51_records:
    for alias, real_sid in r.get('evidence_id_mapping', {}).items():
        if alias == 'E0':
            continue  # E0 is the target session itself, already in target_sids
        clean_sid = real_sid[2:] if real_sid.startswith('E_') else real_sid
        evidence_sids.add(clean_sid)

all_target_sids = target_sids | evidence_sids
print(f"Target sessions:   {len(target_sids)}")
print(f"Evidence sessions: {len(evidence_sids)}")
print(f"Loading HDFS logs for {len(all_target_sids)} sessions total...")

hdfs_loader = HDFSDataLoader(
    log_file=str(PROJECT_ROOT / 'logs' / 'HDFS.log'),
    label_file=str(PROJECT_ROOT / 'logs' / 'anomaly_label_HDFS.csv'),
)
hdfs_loader.load()

sid_to_lines = {}
for s in hdfs_loader.get_sessions():
    if s.session_id in all_target_sids:
        sid_to_lines[s.session_id] = s.lines

found_target  = sum(1 for sid in target_sids   if sid in sid_to_lines)
found_evidence = sum(1 for sid in evidence_sids if sid in sid_to_lines)
print(f"[OK] Target sessions loaded:   {found_target}/{len(target_sids)}")
print(f"[OK] Evidence sessions loaded: {found_evidence}/{len(evidence_sids)}")

missing = all_target_sids - set(sid_to_lines.keys())
if missing:
    print(f"[WARN] Still missing: {missing}")


Target sessions:   24
Evidence sessions: 70
Loading HDFS logs for 94 sessions total...
Loading HDFS logs from: /home/dave/agentic-log-explanations/logs/HDFS.log
Loading labels from: /home/dave/agentic-log-explanations/logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:10, 1058530.43it/s]


Found 575061 unique blocks
[OK] Target sessions loaded:   24/24
[OK] Evidence sessions loaded: 70/70


## 3. Schema & Structural Validation

Verify each explanation has the required JSON structure. This is the *minimum bar* --
passing this does NOT mean the explanation is correct or grounded.

In [4]:
# === Structural validation -- shows this is necessary but NOT sufficient ===

REQUIRED_EXPLANATION_KEYS = {'summary', 'claims', 'signature', 'prediction'}
REQUIRED_CLAIM_KEYS = {'type', 'claim', 'evidence_ids', 'evidence_spans'}
VALID_TYPES = {'observation', 'pattern_match', 'contrast'}

def validate_structure(record):
    """Check JSON schema compliance. Returns (passed: bool, issues: list[str])."""
    issues = []
    exp = record.get('explanation', {})

    # Top-level keys
    missing = REQUIRED_EXPLANATION_KEYS - set(exp.keys())
    if missing:
        issues.append(f"Missing top-level keys: {missing}")

    # Summary non-empty
    if not exp.get('summary', '').strip():
        issues.append("Empty summary")

    # Claims
    claims = exp.get('claims', [])
    if not claims:
        issues.append("No claims")
    for i, c in enumerate(claims):
        c_missing = REQUIRED_CLAIM_KEYS - set(c.keys())
        if c_missing:
            issues.append(f"Claim {i+1}: missing keys {c_missing}")
        if c.get('type') not in VALID_TYPES:
            issues.append(f"Claim {i+1}: invalid type '{c.get('type')}'")
        if not c.get('claim', '').strip():
            issues.append(f"Claim {i+1}: empty claim text")
        if not c.get('evidence_ids'):
            issues.append(f"Claim {i+1}: no evidence_ids")
        if not c.get('evidence_spans'):
            issues.append(f"Claim {i+1}: no evidence_spans")

    # Signature
    sig = exp.get('signature', {})
    if not sig.get('name'):
        issues.append("Missing signature name")

    return len(issues) == 0, issues


print("=" * 70)
print("STRUCTURAL VALIDATION (necessary but NOT sufficient)")
print("=" * 70)
for model_name, records in [("GPT-5.1", gpt51_records), ("llama3.1:8b", llama_records)]:
    passed = sum(1 for r in records if validate_structure(r)[0])
    print(f"\n{model_name}: {passed}/{len(records)} structurally valid")
    for r in records:
        ok, issues = validate_structure(r)
        if not ok:
            print(f"  [FAIL] {r['session_id']}: {issues[:3]}")

STRUCTURAL VALIDATION (necessary but NOT sufficient)

GPT-5.1: 24/24 structurally valid

llama3.1:8b: 20/24 structurally valid
  [FAIL] HDFS_blk_5703197246022264715: ["Claim 3: invalid type 'exception'"]
  [FAIL] HDFS_blk_3338061113250311986: ["Missing top-level keys: {'signature'}", 'No claims', 'Missing signature name']
  [FAIL] HDFS_blk_8095512464329197839: ['No claims']
  [FAIL] HDFS_blk_8239323489440610674: ['No claims']


## 4. Evidence Span Grounding Check

**The core hallucination detector.** For each claim, verify that:
1. Cited evidence IDs (E0, E1...) map to real sessions
2. Cited line references (E0-L17) actually exist in the log
3. The claim text mentions content that genuinely appears in those log lines

A claim that cites `E0-L19` but L19 says something completely different is a **hallucination**.

In [5]:
# === Evidence Span Grounding -- verify claims against actual log lines ===

def parse_span_refs(span_str):
    """Parse 'E0-L17 to E0-L20' or 'E0-L19' into (evidence_id, start_line, end_line)."""
    results = []
    # Pattern: E{n}-L{n} (optionally: to E{n}-L{n})
    range_pat = re.compile(r'(E\d+)-L(\d+)\s+to\s+(E\d+)-L(\d+)')
    single_pat = re.compile(r'(E\d+)-L(\d+)')

    for m in range_pat.finditer(span_str):
        eid_start, l_start, eid_end, l_end = m.groups()
        results.append((eid_start, int(l_start), int(l_end)))

    # Also find standalone single references not part of a range
    already_covered = set()
    for m in range_pat.finditer(span_str):
        already_covered.add(m.span())

    for m in single_pat.finditer(span_str):
        # Check this isn't part of a range we already captured
        is_part_of_range = False
        for (rs, re_) in already_covered:
            if rs <= m.start() and m.end() <= re_:
                is_part_of_range = True
                break
        if not is_part_of_range:
            eid, line_num = m.groups()
            results.append((eid, int(line_num), int(line_num)))

    return results


def get_log_lines_for_evidence(record, evidence_id, sid_to_lines):
    """Get actual log lines for a given evidence ID (E0, E1, etc.)."""
    mapping = record.get('evidence_id_mapping', {})
    real_sid = mapping.get(evidence_id)
    if real_sid is None:
        return None, f"No mapping for {evidence_id}"

    # E0 maps to the session itself, E1+ maps to E_HDFS_blk_xxx (evidence sessions)
    lines = sid_to_lines.get(real_sid)
    if lines is None:
        # Try without E_ prefix
        if real_sid.startswith('E_'):
            lines = sid_to_lines.get(real_sid[2:])
    return lines, real_sid


def check_span_exists(lines, start_line, end_line):
    """Check if line references are within bounds. Lines are 1-indexed."""
    if lines is None:
        return False, "No log lines available"
    if start_line < 1 or end_line > len(lines):
        return False, f"Line range L{start_line}-L{end_line} out of bounds (session has {len(lines)} lines)"
    return True, "OK"


def extract_keywords_from_claim(claim_text):
    """Extract key technical terms from a claim for matching against logs."""
    # Look for quoted strings first
    quoted = re.findall(r"'([^']+)'", claim_text)
    quoted += re.findall(r'"([^"]+)"', claim_text)

    # Also extract key technical terms (uppercase, error-like patterns)
    tech_terms = re.findall(
        r'\b(?:WARN|ERROR|INFO|FATAL|Exception|error|warning|failure|'
        r'delete|block|replication|DataNode|NameNode|FSDataset|'
        r'volumeMap|BlockInfo|PacketResponder|writeBlock|'
        r'received|serving|invalidate)\w*\b',
        claim_text, re.IGNORECASE
    )
    return quoted, list(set(tech_terms))


def check_claim_grounding(claim_text, lines, start_line, end_line):
    """Check if the claim content actually matches the cited log lines."""
    if lines is None:
        return 0.0, "No log lines to check"

    # Get the cited lines (1-indexed)
    cited_lines = lines[start_line-1 : end_line]
    cited_text = ' '.join(cited_lines).lower()

    quoted_terms, tech_terms = extract_keywords_from_claim(claim_text)

    # Score: what fraction of quoted/technical terms appear in the cited lines?
    all_terms = quoted_terms + tech_terms
    if not all_terms:
        return 0.5, "No specific terms to verify"

    matched = sum(1 for t in all_terms if t.lower() in cited_text)
    score = matched / len(all_terms)

    unmatched = [t for t in all_terms if t.lower() not in cited_text]
    detail = f"{matched}/{len(all_terms)} terms matched"
    if unmatched:
        detail += f" | unmatched: {unmatched[:5]}"

    return score, detail


print("[OK] Grounding functions defined")

[OK] Grounding functions defined


## 5. Run Automated Grounding Analysis

Apply span and content grounding checks to all 24 GPT-5.1 explanations.
This gives us a factual consistency score per claim and per session.

In [35]:
# === Full grounding analysis for all GPT-5.1 explanations ===

def analyze_record(record, sid_to_lines):
    """Analyze a single record for grounding quality."""
    sid = record['session_id']
    exp = record['explanation']
    claims = exp.get('claims', [])
    mapping = record.get('evidence_id_mapping', {})

    result = {
        'session_id': sid,
        'signature': record.get('normalized_signature', '?'),
        'n_claims': len(claims),
        'claim_details': [],
        'span_valid': 0,
        'span_invalid': 0,
        'span_total': 0,
        'grounding_scores': [],
        'hallucination_flags': [],
    }

    for i, claim in enumerate(claims):
        claim_text = claim.get('claim', '')
        spans = claim.get('evidence_spans', [])
        ctype = claim.get('type', '?')

        claim_result = {
            'idx': i + 1,
            'type': ctype,
            'claim': claim_text[:200],
            'span_checks': [],
            'grounding_score': 0.0,
            'grounding_detail': '',
            'is_hallucinated': False,
        }

        all_scores = []
        for span_str in spans:
            refs = parse_span_refs(span_str)
            for eid, sl, el in refs:
                result['span_total'] += 1
                lines, real_sid = get_log_lines_for_evidence(record, eid, sid_to_lines)
                exists, exists_msg = check_span_exists(lines, sl, el)

                if exists:
                    result['span_valid'] += 1
                    gscore, gdetail = check_claim_grounding(claim_text, lines, sl, el)
                    all_scores.append(gscore)
                    claim_result['span_checks'].append({
                        'span': span_str, 'ref': f"{eid}-L{sl}..L{el}",
                        'exists': True, 'grounding': gscore, 'detail': gdetail
                    })
                else:
                    result['span_invalid'] += 1
                    all_scores.append(0.0)
                    claim_result['span_checks'].append({
                        'span': span_str, 'ref': f"{eid}-L{sl}..L{el}",
                        'exists': False, 'detail': exists_msg
                    })

        if all_scores:
            claim_result['grounding_score'] = np.mean(all_scores)
        else:
            claim_result['grounding_score'] = 0.0

        # Hallucination flag: low grounding OR invalid spans
        if claim_result['grounding_score'] < 0.3:
            claim_result['is_hallucinated'] = True
            result['hallucination_flags'].append(f"Claim {i+1}: low grounding ({claim_result['grounding_score']:.2f})")

        result['claim_details'].append(claim_result)
        result['grounding_scores'].append(claim_result['grounding_score'])

    # Aggregate
    result['avg_grounding'] = np.mean(result['grounding_scores']) if result['grounding_scores'] else 0.0
    result['span_validity_rate'] = result['span_valid'] / result['span_total'] if result['span_total'] > 0 else 0.0
    result['n_hallucinated_claims'] = sum(1 for c in result['claim_details'] if c['is_hallucinated'])

    return result


# Run analysis
gpt51_analysis = [analyze_record(r, sid_to_lines) for r in gpt51_records]
llama_analysis = [analyze_record(r, sid_to_lines) for r in llama_records]

# Summary
print("=" * 70)
print("AUTOMATED GROUNDING ANALYSIS")
print("=" * 70)

for model_name, analysis in [("GPT-5.1", gpt51_analysis), ("llama3.1:8b", llama_analysis)]:
    avg_ground = np.mean([a['avg_grounding'] for a in analysis])
    avg_span_valid = np.mean([a['span_validity_rate'] for a in analysis])
    total_claims = sum(a['n_claims'] for a in analysis)
    halluc_claims = sum(a['n_hallucinated_claims'] for a in analysis)
    halluc_sessions = sum(1 for a in analysis if a['n_hallucinated_claims'] > 0)

    print(f"\n{model_name}:")
    print(f"  Avg grounding score:    {avg_ground:.3f}")
    print(f"  Avg span validity:      {avg_span_valid:.1%}")
    print(f"  Total claims:           {total_claims}")
    print(f"  Hallucinated claims:    {halluc_claims}/{total_claims} ({halluc_claims/total_claims*100:.1f}%)")
    print(f"  Sessions w/ halluc.:    {halluc_sessions}/{len(analysis)}")

AUTOMATED GROUNDING ANALYSIS

GPT-5.1:
  Avg grounding score:    0.454
  Avg span validity:      89.1%
  Total claims:           72
  Hallucinated claims:    6/72 (8.3%)
  Sessions w/ halluc.:    5/24

llama3.1:8b:
  Avg grounding score:    0.321
  Avg span validity:      80.6%
  Total claims:           37
  Hallucinated claims:    18/37 (48.6%)
  Sessions w/ halluc.:    9/24


## 6. Detailed Claim-Level Report

Show each session's claims with their grounding scores and the actual log lines they cite.
This lets you see exactly what the model claimed vs what the logs actually say.

In [7]:
# === Detailed claim-level grounding report ===

def print_claim_report(analysis_list, model_name, sid_to_lines, records_by_sid,
                       show_all=False, show_logs=True):
    """Print detailed per-claim grounding with actual log lines."""
    W = 90
    print("=" * W)
    print(f"CLAIM-LEVEL GROUNDING REPORT: {model_name}")
    print("=" * W)

    for a in analysis_list:
        sid = a['session_id']
        record = records_by_sid[sid]
        flag = ""
        if a['n_hallucinated_claims'] > 0:
            flag = " [SUSPECT]"
        elif a['avg_grounding'] < 0.5:
            flag = " [LOW GROUNDING]"

        if not show_all and not flag:
            continue  # Skip well-grounded sessions in summary mode

        print(f"\n{'=' * W}")
        print(f"Session: {sid}{flag}")
        print(f"Signature: {a['signature']}")
        print(f"Grounding: {a['avg_grounding']:.2f} | Span validity: {a['span_validity_rate']:.0%} | "
              f"Claims: {a['n_claims']} | Halluc: {a['n_hallucinated_claims']}")
        print(f"Summary: {record['explanation']['summary'][:200]}")

        for cd in a['claim_details']:
            g = cd['grounding_score']
            marker = "[OK]" if g >= 0.5 else "[LOW]" if g >= 0.3 else "[HALLUC?]"
            print(f"\n  Claim {cd['idx']} [{cd['type']}] {marker} grounding={g:.2f}")
            print(f"    {cd['claim']}")

            for sc in cd['span_checks']:
                if sc.get('exists'):
                    print(f"    Span {sc['ref']}: grounding={sc['grounding']:.2f} -- {sc['detail']}")
                else:
                    print(f"    Span {sc['ref']}: [INVALID] {sc['detail']}")

                # Show actual log lines for this span
                if show_logs and sc.get('exists'):
                    ref_parts = re.match(r'(E\d+)-L(\d+)\.\.L(\d+)', sc['ref'])
                    if ref_parts:
                        eid, sl, el = ref_parts.group(1), int(ref_parts.group(2)), int(ref_parts.group(3))
                        lines, _ = get_log_lines_for_evidence(record, eid, sid_to_lines)
                        if lines:
                            print(f"    --- Actual log lines ({eid} L{sl}-L{el}) ---")
                            for ln in range(sl-1, min(el, len(lines))):
                                print(f"      L{ln+1:02d}: {lines[ln][:120]}")

    # Count clean sessions
    clean = sum(1 for a in analysis_list if not a['hallucination_flags'] and a['avg_grounding'] >= 0.5)
    total = len(analysis_list)
    if not show_all:
        print(f"\n{'=' * W}")
        print(f"[OK] {clean}/{total} sessions fully grounded (not shown above)")
        print(f"[FLAG] {total - clean}/{total} sessions shown above with potential issues")


# Show GPT-5.1 flagged sessions with log lines
print_claim_report(gpt51_analysis, "GPT-5.1", sid_to_lines, gpt51_by_sid,
                   show_all=False, show_logs=True)

CLAIM-LEVEL GROUNDING REPORT: GPT-5.1

Session: HDFS_blk_-3661881463166428296 [LOW GROUNDING]
Signature: DATANODE__BLOCK_SERVE_EXCEPTION
Grounding: 0.37 | Span validity: 89% | Claims: 3 | Halluc: 0
Summary: DATANODE__BLOCK_SERVE_EXCEPTION: 4 WARN "Got exception while serving <BLOCK>" events during block reads, followed by invalidation and deletion of the same block at E0-L15 to E0-L25 and E0-L26 to E0-L3

  Claim 1 [observation] [LOW] grounding=0.42
    E0 contains 4 WARN dfs.DataNode$DataXceiver lines with "Got exception while serving <BLOCK>" at E0-L15, E0-L19, E0-L21, and E0-L24, followed by NameSystem.delete and FSDataset "Deleting block" actions
    Span E0-L15..L15: grounding=0.40 -- 4/10 terms matched | unmatched: ['Got exception while serving <BLOCK>', 'Deleting block', 'delete', 'BLOCK', 'FSDataset']
    --- Actual log lines (E0 L15-L15) ---
      L15: 081110 025439 6117 WARN dfs.DataNode$DataXceiver: 10.251.106.10:50010:Got exception while serving blk_-36618814631664282
    S

In [ ]:
# === Show ALL GPT-5.1 sessions (full audit) ===
# Uncomment the line below to see every session including well-grounded ones:
# print_claim_report(gpt51_analysis, "GPT-5.1", sid_to_lines, gpt51_by_sid, show_all=True, show_logs=True)

# === Show llama3.1:8b flagged sessions for comparison ===
print_claim_report(llama_analysis, "llama3.1:8b", sid_to_lines, llama_by_sid,
                   show_all=False, show_logs=True)

## 7. Human Review Interface

Interactive review: display one session at a time with full log lines,
explanation, and grounding analysis. Rate each one using the same rubric as 09.

**Workflow:**
1. Set `IDX` below (0-23), run the display cell
2. Read the log lines, explanation, and grounding flags
3. Fill in scores in the rate cell, run it
4. Increment `IDX` and repeat

In [37]:
# === Human review: display + rate + save (following 09 pattern) ===

# Load or initialize audit ratings
if AUDIT_RATINGS_PATH.exists():
    with open(AUDIT_RATINGS_PATH) as f:
        audit_ratings = json.load(f)
    print(f"[OK] Loaded existing audit ratings from {AUDIT_RATINGS_PATH.name}")
else:
    audit_ratings = {
        'evaluator': 'dave',
        'rubric_version': 'v1_audit',
        'model': 'gpt-5.1',
        'purpose': 'hallucination audit of GPT-5.1 HDFS edge case explanations',
        'dimensions': {
            'correctness': '1-5: Is the explanation factually correct vs the actual logs?',
            'completeness': '1-5: Does it cover all anomaly signals visible in the logs?',
            'evidence_grounding': '1-5: Are claims properly supported by the cited log lines?',
            'actionable': 'Y/N: Could an engineer act on this explanation?',
            'hallucination': '0=none, 1=minor (embellishment), 2=major (fabricated content)',
        },
        'ratings': {},
    }
    print("[OK] Initialized empty audit ratings")


def _audit_progress():
    n = sum(1 for r in gpt51_records if r['session_id'] in audit_ratings['ratings'])
    print(f"  GPT-5.1 audit: {n}/{len(gpt51_records)}")
    unrated = [i for i, r in enumerate(gpt51_records)
               if r['session_id'] not in audit_ratings['ratings']]
    if unrated:
        print(f"  Next unrated: IDX = {unrated[0]}")
    else:
        print(f"  [OK] ALL DONE")


def display_audit(idx, show_llama=True):
    """Display one GPT-5.1 session for audit with logs, explanation, and grounding."""
    assert 0 <= idx < len(gpt51_records), f"IDX must be 0-{len(gpt51_records)-1}"
    record = gpt51_records[idx]
    analysis = gpt51_analysis[idx]
    sid = record['session_id']
    exp = record['explanation']
    rated = '[RATED]' if sid in audit_ratings['ratings'] else '[UNRATED]'

    W = 90
    print('=' * W)
    print(f"[{idx+1}/{len(gpt51_records)}] {sid}  {rated}")
    print(f"Model: GPT-5.1 | Signature: {record.get('normalized_signature', '?')}")
    vp = record['verification']['passed']
    n_issues = len(record['verification'].get('issues', []))
    print(f"Verification: {'PASSED' if vp else 'FAILED'} | Issues: {n_issues}")
    print(f"Grounding score: {analysis['avg_grounding']:.2f} | "
          f"Span validity: {analysis['span_validity_rate']:.0%} | "
          f"Halluc claims: {analysis['n_hallucinated_claims']}/{analysis['n_claims']}")
    print('=' * W)

    # === LOG LINES (E0 = the session itself) ===
    lines = sid_to_lines.get(sid, [])
    print(f"\n--- LOG LINES (E0: {sid}, {len(lines)} lines) ---")
    for i, line in enumerate(lines[:50]):
        print(f"  L{i+1:02d}: {line[:140]}")
    if len(lines) > 50:
        print(f"  ... ({len(lines)} lines total, showing first 50)")

    # === EXPLANATION ===
    print(f"\n--- EXPLANATION ---")
    print(f"  Summary: {exp['summary']}")
    print(f"\n  Claims ({len(exp['claims'])}):")
    for i, c in enumerate(exp['claims']):
        cd = analysis['claim_details'][i]
        g = cd['grounding_score']
        marker = "[OK]" if g >= 0.5 else "[LOW]" if g >= 0.3 else "[HALLUC?]"

        print(f"\n  {i+1}. [{c['type']}] {marker} (grounding={g:.2f})")
        print(f"     {c['claim']}")
        eids = ', '.join(c.get('evidence_ids', []))
        spans_str = ', '.join(c.get('evidence_spans', []))
        print(f"     Evidence: {eids} | Spans: {spans_str}")

        # Show actual log lines for cited spans
        for sc in cd['span_checks']:
            if sc.get('exists'):
                ref_parts = re.match(r'(E\d+)-L(\d+)\.\.L(\d+)', sc['ref'])
                if ref_parts:
                    eid = ref_parts.group(1)
                    sl, el = int(ref_parts.group(2)), int(ref_parts.group(3))
                    elines, _ = get_log_lines_for_evidence(record, eid, sid_to_lines)
                    if elines:
                        print(f"     --- Actual {eid} L{sl}-L{el} ---")
                        for ln in range(sl-1, min(el, len(elines))):
                            print(f"       L{ln+1:02d}: {elines[ln][:130]}")
            else:
                print(f"     [INVALID SPAN] {sc['ref']}: {sc['detail']}")

    # === Evidence ID Mapping ===
    mapping = record.get('evidence_id_mapping', {})
    if mapping:
        print(f"\n  Evidence ID mapping:")
        for alias, real_id in mapping.items():
            print(f"    {alias} -> {real_id}")

    # === LLAMA COMPARISON ===
    if show_llama and sid in llama_by_sid:
        llama_r = llama_by_sid[sid]
        llama_exp = llama_r['explanation']
        print(f"\n--- LLAMA3.1:8b COMPARISON ---")
        print(f"  Verification: {'PASSED' if llama_r['verification']['passed'] else 'FAILED'}")
        print(f"  Summary: {llama_exp['summary'][:200]}")
        print(f"  Claims: {len(llama_exp['claims'])}")
        for i, c in enumerate(llama_exp['claims'][:3]):
            print(f"    {i+1}. [{c['type']}] {c['claim'][:120]}")

    print('\n' + '=' * W)
    _audit_progress()


def rate_audit(idx, correctness, completeness, evidence_grounding,
               actionable, hallucination=0, notes=''):
    """Rate a GPT-5.1 session and save to disk."""
    sid = gpt51_records[idx]['session_id']

    for name, val in [('correctness', correctness),
                      ('completeness', completeness),
                      ('evidence_grounding', evidence_grounding)]:
        assert isinstance(val, int) and 1 <= val <= 5, \
            f"{name} must be int 1-5, got {val}"
    assert actionable in ('Y', 'N'), f"actionable must be 'Y' or 'N'"
    assert hallucination in (0, 1, 2), "hallucination must be 0, 1, or 2"

    audit_ratings['ratings'][sid] = {
        'idx': idx,
        'normalized_signature': gpt51_records[idx].get('normalized_signature', '?'),
        'correctness': correctness,
        'completeness': completeness,
        'evidence_grounding': evidence_grounding,
        'actionable': actionable,
        'hallucination': hallucination,
        'auto_grounding': gpt51_analysis[idx]['avg_grounding'],
        'notes': notes,
        'rated_at': datetime.now().isoformat(),
    }

    with open(AUDIT_RATINGS_PATH, 'w') as f:
        json.dump(audit_ratings, f, indent=2, ensure_ascii=False)

    n = sum(1 for r in gpt51_records if r['session_id'] in audit_ratings['ratings'])
    print(f"[OK] {sid}: C={correctness} Co={completeness} E={evidence_grounding} "
          f"A={actionable} H={hallucination}")
    print(f"     Progress: {n}/{len(gpt51_records)}")


print("\nCurrent progress:")
_audit_progress()
print("\n[OK] display_audit() and rate_audit() ready")

[OK] Loaded existing audit ratings from gpt51_audit_ratings.json

Current progress:
  GPT-5.1 audit: 11/24
  Next unrated: IDX = 11

[OK] display_audit() and rate_audit() ready


In [ ]:
# === DISPLAY: change IDX and run ===
IDX = 11  # 0-23

display_audit(IDX, show_llama=True)


[11/24] HDFS_blk_1340637939534925227  [UNRATED]
Model: GPT-5.1 | Signature: DATANODE__SERVE_BLOCK_EXCEPTION_AND_REDUNDANT_INVALIDATION
Verification: PASSED | Issues: 9
Grounding score: 0.36 | Span validity: 88% | Halluc claims: 1/3

--- LOG LINES (E0: HDFS_blk_1340637939534925227, 33 lines) ---
  L01: 081110 012211 28 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /user/root/randtxt/_temporary/_task_200811092030_0003_m_000803_0/pa
  L02: 081110 012211 5110 INFO dfs.DataNode$DataXceiver: Receiving block blk_1340637939534925227 src: /10.251.106.10:42883 dest: /10.251.106.10:500
  L03: 081110 012211 5409 INFO dfs.DataNode$DataXceiver: Receiving block blk_1340637939534925227 src: /10.251.38.53:56017 dest: /10.251.38.53:50010
  L04: 081110 012211 5411 INFO dfs.DataNode$DataXceiver: Receiving block blk_1340637939534925227 src: /10.251.106.10:41777 dest: /10.251.106.10:500
  L05: 081110 012244 28 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.106.10

In [ ]:
# === RATE: fill in scores and run ===
rate_audit(IDX,
    correctness=5,           # 1-5: factual accuracy vs logs
    completeness=5,          # 1-5: covers all anomaly signals
    evidence_grounding=5,    # 1-5: claims match cited log lines (E1-E5 now loadable)
    actionable='Y',          # 'Y' or 'N'
    hallucination=0,         # 0=none, 1=minor embellishment, 2=major fabrication
    notes='',
)


[OK] HDFS_blk_1340637939534925227: C=5 Co=5 E=4 A=Y H=0
     Progress: 11/24


## 8. Aggregate Scores & Flag Suspicious Explanations

Summary DataFrame with automated + human scores. Flag cases where:
- Verification PASSED but grounding is low (structurally gamed)
- High auto-eval scores but human rates as hallucinated

In [11]:
# === Aggregate: automated grounding + human ratings ===

rows = []
for i, (record, analysis) in enumerate(zip(gpt51_records, gpt51_analysis)):
    sid = record['session_id']
    row = {
        'idx': i,
        'session_id': sid,
        'signature': record.get('normalized_signature', '?'),
        'verif_passed': record['verification']['passed'],
        'n_claims': analysis['n_claims'],
        'auto_grounding': analysis['avg_grounding'],
        'span_validity': analysis['span_validity_rate'],
        'n_halluc_claims': analysis['n_hallucinated_claims'],
    }

    # Add human ratings if available
    if sid in audit_ratings['ratings']:
        r = audit_ratings['ratings'][sid]
        row['human_correctness'] = r['correctness']
        row['human_completeness'] = r['completeness']
        row['human_evidence'] = r['evidence_grounding']
        row['human_actionable'] = 1 if r['actionable'] == 'Y' else 0
        row['human_halluc'] = r['hallucination']
        row['notes'] = r.get('notes', '')
    rows.append(row)

df = pd.DataFrame(rows)

# === Summary ===
print("=" * 80)
print("AGGREGATE QUALITY REPORT: GPT-5.1 (24 HDFS Edge Cases)")
print("=" * 80)

print(f"\n--- Automated Grounding ---")
print(f"  Mean grounding score:    {df['auto_grounding'].mean():.3f}")
print(f"  Mean span validity:      {df['span_validity'].mean():.1%}")
print(f"  Sessions w/ halluc. claims: {(df['n_halluc_claims'] > 0).sum()}/{len(df)}")
print(f"  Total halluc. claims:    {df['n_halluc_claims'].sum()}/{df['n_claims'].sum()}")

# Human ratings (if any)
human_cols = ['human_correctness', 'human_completeness', 'human_evidence']
has_human = all(c in df.columns for c in human_cols)
rated_df = df.dropna(subset=['human_correctness']) if has_human else df.iloc[:0]
if len(rated_df) > 0:
    print(f"\n--- Human Ratings ({len(rated_df)}/{len(df)} rated) ---")
    for dim in ['human_correctness', 'human_completeness', 'human_evidence']:
        label = dim.replace('human_', '')
        print(f"  {label:<20} mean={rated_df[dim].mean():.2f}  "
              f"median={rated_df[dim].median():.1f}  "
              f"min={rated_df[dim].min():.0f}  max={rated_df[dim].max():.0f}")
    print(f"  {'actionable':<20} {rated_df['human_actionable'].mean():.0%}")
    print(f"  {'hallucination=0':<20} {(rated_df['human_halluc'] == 0).sum()}/{len(rated_df)}")
    print(f"  {'hallucination=1':<20} {(rated_df['human_halluc'] == 1).sum()}/{len(rated_df)} (minor)")
    print(f"  {'hallucination=2':<20} {(rated_df['human_halluc'] == 2).sum()}/{len(rated_df)} (major)")

    # === Flag suspicious: verif PASSED but human says hallucinated ===
    flagged = rated_df[(rated_df['verif_passed'] == True) & (rated_df['human_halluc'] > 0)]
    if len(flagged) > 0:
        print(f"\n[FLAG] {len(flagged)} sessions PASS verification but have hallucinations:")
        for _, row in flagged.iterrows():
            print(f"  {row['session_id']}: halluc={row['human_halluc']}, "
                  f"C={row['human_correctness']}, notes={row.get('notes','')}")
    else:
        print(f"\n[OK] No sessions flagged as 'verification passed + hallucinated'")

    # === Auto vs Human agreement ===
    print(f"\n--- Auto Grounding vs Human Evidence Score ---")
    corr = rated_df[['auto_grounding', 'human_evidence']].corr().iloc[0, 1]
    print(f"  Pearson correlation: {corr:.3f}")
else:
    print(f"\n[INFO] No human ratings yet. Rate sessions in Section 7 first.")

# Show full table
print(f"\n--- Per-Session Summary ---")
display_cols = ['idx', 'signature', 'auto_grounding', 'span_validity', 'n_halluc_claims']
if len(rated_df) > 0:
    display_cols += ['human_correctness', 'human_evidence', 'human_halluc']
print(df[display_cols].to_string(index=False))

AGGREGATE QUALITY REPORT: GPT-5.1 (24 HDFS Edge Cases)

--- Automated Grounding ---
  Mean grounding score:    0.408
  Mean span validity:      73.8%
  Sessions w/ halluc. claims: 13/24
  Total halluc. claims:    16/72

[INFO] No human ratings yet. Rate sessions in Section 7 first.

--- Per-Session Summary ---
 idx                                                     signature  auto_grounding  span_validity  n_halluc_claims
   0                   FSDATASET__BLOCK_DELETION_METADATA_MISMATCH        0.770655       0.750000                0
   1                               DATANODE__BLOCK_SERVE_EXCEPTION        0.367003       0.888889                0
   2                               DATANODE__SERVE_BLOCK_EXCEPTION        0.451515       0.600000                0
   3                               DATANODE__BLOCK_SERVE_EXCEPTION        0.558201       0.600000                0
   4                               DATANODE__SERVE_BLOCK_EXCEPTION        0.356349       0.833333                

## 9. Visualization

Grayscale plots comparing automated grounding scores, human ratings,
and hallucination distribution.

In [12]:
# === Visualization (grayscale, 600 dpi) ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Plot 1: Grounding score distribution (GPT-5.1 vs llama) ---
ax = axes[0]
gpt_scores = [a['avg_grounding'] for a in gpt51_analysis]
llama_scores = [a['avg_grounding'] for a in llama_analysis]
bins = np.arange(0, 1.05, 0.1)
ax.hist(gpt_scores, bins=bins, alpha=0.7, color='0.3', label='GPT-5.1',
        edgecolor='0.1', linewidth=0.8)
ax.hist(llama_scores, bins=bins, alpha=0.5, color='0.7', label='llama3.1:8b',
        edgecolor='0.4', linewidth=0.8, linestyle='--')
ax.set_xlabel('Avg Grounding Score')
ax.set_ylabel('Count')
ax.set_title('Claim Grounding Distribution')
ax.legend(frameon=False)

# --- Plot 2: Per-session grounding + span validity ---
ax = axes[1]
x = range(len(gpt51_analysis))
ground = [a['avg_grounding'] for a in gpt51_analysis]
span_v = [a['span_validity_rate'] for a in gpt51_analysis]
ax.bar(x, ground, width=0.4, color='0.3', label='Grounding', align='center')
ax.bar([i+0.4 for i in x], span_v, width=0.4, color='0.7', label='Span Validity',
       align='center')
ax.set_xlabel('Session Index')
ax.set_ylabel('Score')
ax.set_title('GPT-5.1 Per-Session Quality')
ax.legend(frameon=False, fontsize=8)
ax.set_ylim(0, 1.1)

# --- Plot 3: Human ratings distribution (if available) ---
ax = axes[2]
has_human_viz = all(c in df.columns for c in ['human_correctness', 'human_completeness', 'human_evidence'])
rated_df_viz = df.dropna(subset=['human_correctness']) if has_human_viz else df.iloc[:0]
if len(rated_df_viz) > 0:
    dims = ['human_correctness', 'human_completeness', 'human_evidence']
    labels = ['Correctness', 'Completeness', 'Evidence']
    positions = np.arange(len(dims))
    means = [rated_df_viz[d].mean() for d in dims]
    stds = [rated_df_viz[d].std() for d in dims]
    ax.bar(positions, means, yerr=stds, width=0.6, color='0.4', edgecolor='0.1',
           capsize=4)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=15)
    ax.set_ylabel('Score (1-5)')
    ax.set_title('Human Ratings')
    ax.set_ylim(0, 5.5)
    # Add hallucination count annotation
    h0 = (rated_df_viz['human_halluc'] == 0).sum()
    h1 = (rated_df_viz['human_halluc'] == 1).sum()
    h2 = (rated_df_viz['human_halluc'] == 2).sum()
    ax.text(0.95, 0.95, f"Halluc: none={h0} minor={h1} major={h2}",
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round', facecolor='0.9', alpha=0.8))
else:
    ax.text(0.5, 0.5, 'Rate sessions in\nSection 7 first',
            transform=ax.transAxes, ha='center', va='center', fontsize=11)
    ax.set_title('Human Ratings (pending)')

plt.tight_layout()
out_path = PROJECT_ROOT / 'results_HDFS' / 'gpt51_audit_quality.png'
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {out_path.name}")

[OK] Saved: gpt51_audit_quality.png


/tmp/ipykernel_1045359/2767350777.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. GPT-5.1 vs llama3.1:8b Side-by-Side Comparison

Compare both models' explanations for the same session to see qualitative differences.
Useful for spotting where GPT-5.1 improves vs where it might over-generate.

In [ ]:
# === Side-by-side comparison for a specific session ===

def compare_models(idx):
    """Show GPT-5.1 and llama3.1:8b explanations side-by-side for same session."""
    gpt_r = gpt51_records[idx]
    sid = gpt_r['session_id']
    llama_r = llama_by_sid.get(sid)
    gpt_a = gpt51_analysis[idx]

    W = 90
    lines = sid_to_lines.get(sid, [])

    print("=" * W)
    print(f"SIDE-BY-SIDE: Session {idx+1}/24 -- {sid}")
    print("=" * W)

    # Log lines (shared)
    print(f"\n--- LOG LINES ({len(lines)} lines) ---")
    for i, line in enumerate(lines[:25]):
        print(f"  L{i+1:02d}: {line[:130]}")
    if len(lines) > 25:
        print(f"  ... ({len(lines)} total)")

    # GPT-5.1
    gpt_exp = gpt_r['explanation']
    print(f"\n{'=' * W}")
    print(f"GPT-5.1 | Sig: {gpt_r.get('normalized_signature','?')} | "
          f"Verif: {'PASS' if gpt_r['verification']['passed'] else 'FAIL'} | "
          f"Grounding: {gpt_a['avg_grounding']:.2f}")
    print(f"  Summary: {gpt_exp['summary']}")
    for i, c in enumerate(gpt_exp['claims']):
        cd = gpt_a['claim_details'][i]
        g = cd['grounding_score']
        print(f"  {i+1}. [{c['type']}] (g={g:.2f}) {c['claim'][:120]}")
        print(f"     Spans: {', '.join(c.get('evidence_spans',[]))}")

    # llama
    if llama_r:
        llama_idx = next(i for i, r in enumerate(llama_records) if r['session_id'] == sid)
        llama_a = llama_analysis[llama_idx]
        llama_exp = llama_r['explanation']
        print(f"\n{'-' * W}")
        print(f"llama3.1:8b | Sig: {llama_r.get('normalized_signature','?')} | "
              f"Verif: {'PASS' if llama_r['verification']['passed'] else 'FAIL'} | "
              f"Grounding: {llama_a['avg_grounding']:.2f}")
        print(f"  Summary: {llama_exp['summary']}")
        for i, c in enumerate(llama_exp['claims']):
            cd = llama_a['claim_details'][i]
            g = cd['grounding_score']
            print(f"  {i+1}. [{c['type']}] (g={g:.2f}) {c['claim'][:120]}")
            print(f"     Spans: {', '.join(c.get('evidence_spans',[]))}")
    else:
        print(f"\n[INFO] No llama result for this session")

    print("=" * W)

# Compare session 0
compare_models(0)

## 11. Final Verdict

Run after completing human ratings for all 24 sessions.
Produces a final quality assessment and paper-ready summary table.

In [13]:
# === Final Assessment ===

has_human_final = all(c in df.columns for c in ['human_correctness', 'human_completeness', 'human_evidence'])
rated_df = df.dropna(subset=['human_correctness']) if has_human_final else df.iloc[:0]
n_rated = len(rated_df)

print("=" * 80)
print("FINAL QUALITY ASSESSMENT: GPT-5.1 HDFS Edge Cases")
print("=" * 80)

if n_rated < len(gpt51_records):
    print(f"\n[WARN] Only {n_rated}/{len(gpt51_records)} sessions rated. "
          f"Complete rating in Section 7 for full assessment.")

if n_rated > 0:
    DIMS = ['human_correctness', 'human_completeness', 'human_evidence']
    LABELS = ['Correctness', 'Completeness', 'Evidence Grounding']

    print(f"\n{'Dimension':<25} {'Mean':>6} {'Median':>8} {'Std':>6} {'Min':>5} {'Max':>5}")
    print("-" * 60)
    for dim, label in zip(DIMS, LABELS):
        v = rated_df[dim]
        print(f"{label:<25} {v.mean():>6.2f} {v.median():>8.1f} {v.std():>6.2f} "
              f"{v.min():>5.0f} {v.max():>5.0f}")
    print(f"{'Actionable':<25} {rated_df['human_actionable'].mean():>6.0%}")

    # Hallucination summary
    h_counts = rated_df['human_halluc'].value_counts().sort_index()
    print(f"\nHallucination Assessment:")
    for level, label in [(0, 'None'), (1, 'Minor (embellishment)'), (2, 'Major (fabrication)')]:
        cnt = h_counts.get(level, 0)
        pct = cnt / n_rated * 100
        print(f"  {label:<30} {cnt:>3}/{n_rated} ({pct:.0f}%)")

    # Auto vs Human correlation
    print(f"\nAutomated vs Human Agreement:")
    print(f"  Auto grounding vs Human evidence: r = "
          f"{rated_df[['auto_grounding', 'human_evidence']].corr().iloc[0,1]:.3f}")

    # Verdict
    avg_c = rated_df['human_correctness'].mean()
    avg_e = rated_df['human_evidence'].mean()
    pct_no_halluc = (rated_df['human_halluc'] == 0).mean()

    print(f"\n{'=' * 80}")
    if avg_c >= 4.0 and avg_e >= 4.0 and pct_no_halluc >= 0.8:
        print("[VERDICT] GPT-5.1 explanations are GENUINE -- high quality, well grounded")
    elif avg_c >= 3.0 and pct_no_halluc >= 0.6:
        print("[VERDICT] GPT-5.1 explanations are PARTIALLY RELIABLE -- some grounding issues")
    else:
        print("[VERDICT] GPT-5.1 explanations show SIGNIFICANT HALLUCINATION risk")
    print(f"  Correctness={avg_c:.2f}/5, Evidence={avg_e:.2f}/5, "
          f"No-halluc={pct_no_halluc:.0%}")
    print("=" * 80)
else:
    print("\n[INFO] Rate sessions in Section 7 to get final verdict.")

FINAL QUALITY ASSESSMENT: GPT-5.1 HDFS Edge Cases

[WARN] Only 0/24 sessions rated. Complete rating in Section 7 for full assessment.

[INFO] Rate sessions in Section 7 to get final verdict.
